In [1]:
import os
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from numpy import NaN
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


import numpy as np
import pandas as pd

from sklearn import preprocessing
from sklearn.model_selection import train_test_split

2023-02-10 13:35:26.080660: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-02-10 13:35:26.201806: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-02-10 13:35:26.728587: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory
2023-02-10 13:35:26.728650: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] 

In [2]:
url = 'IoT Network Intrusion Dataset.csv'
data=pd.read_csv(url)
data.shape

(625783, 86)

In [3]:
data.head(5)

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Label,Cat,Sub_Cat
0,192.168.0.13-192.168.0.16-10000-10101-17,192.168.0.13,10000,192.168.0.16,10101,17,25/07/2019 03:25:53 AM,75,1,1,...,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Anomaly,Mirai,Mirai-Ackflooding
1,192.168.0.13-222.160.179.132-554-2179-6,222.160.179.132,2179,192.168.0.13,554,6,26/05/2019 10:11:06 PM,5310,1,2,...,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,Anomaly,DoS,DoS-Synflooding
2,192.168.0.13-192.168.0.16-9020-52727-6,192.168.0.16,52727,192.168.0.13,9020,6,11/07/2019 01:24:48 AM,141,0,3,...,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Anomaly,Scan,Scan Port OS
3,192.168.0.13-192.168.0.16-9020-52964-6,192.168.0.16,52964,192.168.0.13,9020,6,04/09/2019 03:58:17 AM,151,0,2,...,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Anomaly,Mirai,Mirai-Hostbruteforceg
4,192.168.0.1-239.255.255.250-36763-1900-17,192.168.0.1,36763,239.255.255.250,1900,17,10/09/2019 01:41:18 AM,153,2,1,...,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Anomaly,Mirai,Mirai-Hostbruteforceg


In [4]:
# remove attribute 'difficulty_level'
data.drop(['Label','Sub_Cat'],axis=1,inplace=True)
data.shape

(625783, 84)

In [5]:
# number of attack labels 
data['Cat'].value_counts()

Mirai                415677
Scan                  75265
DoS                   59391
Normal                40073
MITM ARP Spoofing     35377
Name: Cat, dtype: int64

In [6]:
dataset = data['Cat']


In [7]:
# Checking if there are any NULL values in the dataset.

data.isnull().values.any()

False

In [8]:
# Checking which column/s contain NULL values.

[col for col in data if data[col].isnull().values.any()]

[]

In [9]:
from sklearn.preprocessing import LabelEncoder

ord_enc = LabelEncoder()
data["Flow_ID"] = ord_enc.fit_transform(data[["Flow_ID"]])
data["Src_IP"] = ord_enc.fit_transform(data[["Src_IP"]])
data["Dst_IP"] = ord_enc.fit_transform(data[["Dst_IP"]])
data["Timestamp"] = ord_enc.fit_transform(data[["Timestamp"]])

/home/mansi/anaconda3/envs/Tensorflow/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/mansi/anaconda3/envs/Tensorflow/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/mansi/anaconda3/envs/Tensorflow/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/mansi/anaconda3/envs/Tensorflow/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:115: DataConversionWar

In [10]:
# Drop the repeated rows
df=data.drop_duplicates()

In [11]:
labl = df['Cat']
dataset = df.loc[:, df.columns != 'Cat'].astype('float64')

In [12]:
# Checking if all values are finite.

np.all(np.isfinite(dataset))

False

In [13]:
# Checking what column/s contain non-finite values.

nonfinite = [col for col in dataset if not np.all(np.isfinite(dataset[col]))]

nonfinite

['Flow_Byts/s', 'Flow_Pkts/s']

In [14]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['Flow_Byts/s']).sum()

dataset.shape[0] - finite

323

In [15]:
# Checking how many non-finite values each column contains.

finite = np.isfinite(dataset['Flow_Pkts/s']).sum()

dataset.shape[0] - finite

323

In [16]:
# Same as before, since there is a small number of non-finite values we can safely remove them from the dataset
# without spoiling the dataset.

# Replacing infinite values with NaN values.
dataset = dataset.replace([np.inf, -np.inf], np.nan)

In [17]:
# We can see that now we have Nan values again.

np.any(np.isnan(dataset))

True

In [18]:
# Bringing the Labels back into the dataset before deliting Nan rows.

dataset = dataset.merge(labl, how='outer', left_index=True, right_index=True)

In [19]:
# Removing new NaN values.

dataset.dropna(inplace=True)

In [20]:
dataset.shape

(411382, 84)

In [21]:
dataset.head()

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Fwd_Seg_Size_Min,Active_Mean,Active_Std,Active_Max,Active_Min,Idle_Mean,Idle_Std,Idle_Max,Idle_Min,Cat
0,12446.0,25883.0,10000.0,203.0,10101.0,17.0,3496.0,75.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,75.0,0.000000,75.0,75.0,Mirai
1,22760.0,34617.0,2179.0,200.0,554.0,6.0,3664.0,5310.0,1.0,2.0,...,0.0,0.0,0.0,0.0,0.0,2655.0,2261.327486,4254.0,1056.0,DoS
2,12691.0,25886.0,52727.0,200.0,9020.0,6.0,2082.0,141.0,0.0,3.0,...,0.0,0.0,0.0,0.0,0.0,70.5,0.707107,71.0,70.0,Scan
3,12704.0,25886.0,52964.0,200.0,9020.0,6.0,791.0,151.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,151.0,0.000000,151.0,151.0,Mirai
4,611.0,25881.0,36763.0,317.0,1900.0,17.0,1040.0,153.0,2.0,1.0,...,0.0,0.0,0.0,0.0,0.0,76.5,0.707107,77.0,76.0,Mirai


In [22]:
dataset['Cat'].value_counts()

Mirai                230788
DoS                   59390
Scan                  56744
Normal                38598
MITM ARP Spoofing     25862
Name: Cat, dtype: int64

In [23]:
# using standard scaler for normalizing
from sklearn.preprocessing import StandardScaler

std_scaler = StandardScaler()
def normalization(df,col):
  for i in col:
    arr = df[i]
    arr = np.array(arr)
    df[i] = std_scaler.fit_transform(arr.reshape(len(arr),1))
  return df

In [24]:
# creating a dataframe with multi-class labels
from sklearn import preprocessing
numeric_col = dataset.select_dtypes(include='number').columns
dataset = normalization(dataset.copy(),numeric_col)


In [25]:
multi_data = dataset.copy()
multi_label = pd.DataFrame(multi_data.Cat)

In [26]:
# label encoding
le2_test = preprocessing.LabelEncoder()
enc_label = multi_label.apply(le2_test.fit_transform)
multi_data['intrusion'] = enc_label

In [27]:
# one-hot-encoding attack label
multi_data = pd.get_dummies(multi_data,columns=['Cat'],prefix="",prefix_sep="") 
multi_data['Cat'] = multi_label
multi_data

,Flow_ID,Src_IP,Src_Port,Dst_IP,Dst_Port,Protocol,Timestamp,Flow_Duration,Tot_Fwd_Pkts,Tot_Bwd_Pkts,...,Idle_Std,Idle_Max,Idle_Min,intrusion,DoS,MITM ARP Spoofing,Mirai,Normal,Scan,Cat
0,-0.709812,0.073387,-1.097955,-0.159136,-0.297670,1.385704,0.861750,-0.189646,-0.196043,-0.354896,...,-0.053756,-0.204341,-0.245423,2,0,0,1,0,0,Mirai
1,-0.232045,1.061937,-1.418388,-0.226262,-0.851415,-0.694809,1.022992,1.030456,-0.196043,0.354889,...,1.536877,0.985235,0.170110,0,1,0,0,0,0,DoS
2,-0.698463,0.073727,0.652609,-0.226262,-0.360371,-0.694809,-0.495373,-0.174264,-0.386957,1.064674,...,-0.053258,-0.205480,-0.247541,4,0,0,0,0,1,Scan
3,-0.697861,0.073727,0.662319,-0.226262,-0.360371,-0.694809,-1.734444,-0.171933,-0.386957,0.354889,...,-0.053756,-0.182707,-0.213231,2,0,0,1,0,0,Mirai
4,-1.258035,0.073161,-0.001451,2.391645,-0.773345,1.385704,-1.495460,-0.171467,-0.005130,-0.354896,...,-0.053258,-0.203772,-0.245000,2,0,0,1,0,0,Mirai
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625773,1.605972,0.074066,0.957350,0.512122,-0.367389,1.385704,0.620846,-0.200367,0.567610,-0.354896,...,-0.051402,-0.222559,-0.275921,2,0,0,1,0,0,Mirai
625776,1.440694,-0.475442,-1.149619,-0.114385,0.253523,-0.694809,-1.980146,0.047382,-0.386957,0.354889,...,-0.053756,0.085154,0.185359,0,1,0,0,0,0,DoS
625778,1.589389,0.074066,0.791295,0.512122,-0.417038,1.385704,0.810881,-0.142567,-0.196043,-0.354896,...,-0.053756,-0.146841,-0.159860,2,0,0,1,0,0,Mirai
625779,-0.417334,0.609880,-1.320427,-0.226262,-0.851415,-0.694809,0.997078,0.179298,-0.386957,0.354889,...,-0.053756,0.246269,0.425106,0,1,0,0,0,0,DoS


In [28]:
# creating a dataframe with only numeric attributes of multi-class dataset and encoded label attribute


numeric_multi = multi_data[numeric_col]
numeric_multi['intrusion'] = multi_data['intrusion']

/tmp/ipykernel_19916/856718132.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  numeric_multi['intrusion'] = multi_data['intrusion']


In [29]:
# finding the attributes which have more than 0.5 correlation with encoded attack label attribute 
corr = numeric_multi.corr()
corr_y = abs(corr['intrusion'])
highest_corr = corr_y[corr_y >0.2]
highest_corr.sort_values(ascending=True)

Pkt_Len_Min         0.206749
Pkt_Len_Var         0.232915
Pkt_Len_Std         0.233685
Bwd_Pkt_Len_Min     0.262157
Src_Port            0.299144
Flow_Duration       0.310543
Pkt_Size_Avg        0.311489
Bwd_Pkt_Len_Mean    0.313733
Bwd_Seg_Size_Avg    0.313733
Pkt_Len_Mean        0.316520
Bwd_IAT_Tot         0.328604
Bwd_Pkt_Len_Max     0.333577
Pkt_Len_Max         0.335450
Dst_Port            0.341456
Idle_Max            0.346567
Flow_IAT_Max        0.349534
Bwd_IAT_Max         0.359086
Bwd_IAT_Min         0.399313
Bwd_IAT_Mean        0.404231
Idle_Mean           0.419217
Idle_Min            0.425997
Flow_IAT_Mean       0.445886
Flow_IAT_Min        0.447646
ACK_Flag_Cnt        0.528705
SYN_Flag_Cnt        0.718676
intrusion           1.000000
Name: intrusion, dtype: float64

In [30]:
# selecting attributes found by using pearson correlation coefficient
numeric_multi = multi_data[['Pkt_Len_Min','Pkt_Len_Var','Pkt_Len_Std','Bwd_Pkt_Len_Min','Src_Port','Flow_Duration',
                            'Pkt_Size_Avg','Bwd_Pkt_Len_Mean','Bwd_Seg_Size_Avg','Pkt_Len_Mean','Bwd_IAT_Tot',
                            'Bwd_Pkt_Len_Max','Pkt_Len_Max','Dst_Port','Idle_Max','Flow_IAT_Max','Bwd_IAT_Max',
                            'Bwd_IAT_Min','Bwd_IAT_Mean','Idle_Mean','Idle_Min','Flow_IAT_Mean','Flow_IAT_Min',
                            'ACK_Flag_Cnt','SYN_Flag_Cnt']]

# then joining encoded, one-hot-encoded, and original attack label attribute
multi_data = numeric_multi.join(multi_data[['intrusion','Mirai','DoS','Scan','Normal','MITM ARP Spoofing','Cat']])

In [31]:
multi_data

,Pkt_Len_Min,Pkt_Len_Var,Pkt_Len_Std,Bwd_Pkt_Len_Min,Src_Port,Flow_Duration,Pkt_Size_Avg,Bwd_Pkt_Len_Mean,Bwd_Seg_Size_Avg,Pkt_Len_Mean,...,Flow_IAT_Min,ACK_Flag_Cnt,SYN_Flag_Cnt,intrusion,Mirai,DoS,Scan,Normal,MITM ARP Spoofing,Cat
0,1.137632,-0.013904,0.649157,1.666777,-1.097955,-0.189646,1.455627,1.580407,1.580407,1.376925,...,-0.258419,-1.018921,-0.405749,2,1,0,0,0,0,Mirai
1,-0.585187,-0.385863,-0.413158,-0.654467,-1.418388,1.030456,-0.751558,-0.741622,-0.741622,-0.756864,...,0.208920,-1.018921,2.464575,0,0,1,0,0,0,DoS
2,-0.532556,2.177439,2.375569,-0.605769,0.652609,-0.174264,0.854712,0.777169,0.777169,0.990099,...,-0.260800,0.981431,-0.405749,4,0,0,1,0,0,Scan
3,1.849918,-0.385863,-0.413158,1.598600,0.662319,-0.171933,1.640613,1.512208,1.512208,1.555759,...,-0.222213,0.981431,-0.405749,2,1,0,0,0,0,Mirai
4,0.151660,-0.384583,-0.350826,0.027297,-0.001451,-0.171467,-0.090513,-0.059627,-0.059627,-0.037918,...,-0.257942,-1.018921,-0.405749,2,1,0,0,0,0,Mirai
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
625773,-0.529047,-0.385863,-0.413158,-0.602523,0.957350,-0.200367,-0.708663,-0.689660,-0.689660,-0.703547,...,-0.292719,-1.018921,-0.405749,2,1,0,0,0,0,Mirai
625776,-0.585187,-0.385863,-0.413158,-0.654467,-1.149619,0.047382,-0.751558,-0.741622,-0.741622,-0.756864,...,0.226070,-1.018921,2.464575,0,0,1,0,0,0,DoS
625778,-0.553608,-0.385863,-0.413158,-0.625248,0.791295,-0.142567,-0.720536,-0.712393,-0.712393,-0.726873,...,-0.162188,-1.018921,-0.405749,2,1,0,0,0,0,Mirai
625779,-0.585187,-0.385863,-0.413158,-0.654467,-1.320427,0.179298,-0.751558,-0.741622,-0.741622,-0.756864,...,0.495707,-1.018921,2.464575,0,0,1,0,0,0,DoS


In [32]:
# classifier for multiclass classification dataset
import warnings
warnings.filterwarnings("ignore")

X = multi_data.iloc[:,0:25].to_numpy() # dataset excluding target attribute (encoded, one-hot-encoded,original)
y = multi_data['intrusion'] # target attribute

X_train, X_validation, Y_train, Y_validation = train_test_split(X, y, test_size=0.20, random_state=1)


In [33]:
X_train = preprocessing.scale(X_train)
X_train = preprocessing.normalize(X_train)

In [34]:
X_validation = preprocessing.scale(X_validation)
X_validation = preprocessing.normalize(X_validation)

In [35]:
print(len(X_train), "Training sequences",X_train.shape)
print(len(X_validation), "Validation sequences",X_validation.shape)

329105 Training sequences (329105, 25)
82277 Validation sequences (82277, 25)


In [36]:
X_train = np.reshape(X_train,(X_train.shape[0],X_train.shape[1],1))
X_validation = np.reshape(X_validation,(X_validation.shape[0],X_validation.shape[1],1))

In [37]:
X_train.shape

(329105, 25, 1)

In [39]:
#Using tanh and sigmoid as activation functions

import time

Model = keras.Sequential([

        keras.layers.Conv2D(96,(4,4),input_shape=(X_train.shape[1],X_train.shape[2],1),activation='relu',padding='same'),
        keras.layers.Conv2D(64,(3,3),activation="relu",padding='same'),
        keras.layers.Conv2D(32,(2,2),activation="relu",padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512,activation="relu"),
        keras.layers.Dense(128,activation="relu"),
        keras.layers.Dense(32,activation="relu"),
        keras.layers.Dense(5,activation="softmax"),
    
    
    ])

Model.compile(optimizer='adam',loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])
start_time = time.time()
#Training the model
Model.fit(X_train, Y_train, epochs=5, batch_size=64) 
Model.summary()

# Final evaluation of the model
scores = Model.evaluate(X_validation, Y_validation, verbose=0)
delta = time.time()- start_time
print("Accuracy: %.2f%%" % (scores[1]*100))
print("Training time: %.2f sec" % (delta))

Epoch 1/5
5143/5143 [==============================] - 46s 9ms/step - loss: 0.3444 - sparse_categorical_accuracy: 0.8622
Epoch 2/5
5143/5143 [==============================] - 45s 9ms/step - loss: 0.2524 - sparse_categorical_accuracy: 0.9018
Epoch 3/5
5143/5143 [==============================] - 45s 9ms/step - loss: 0.2309 - sparse_categorical_accuracy: 0.9116
Epoch 4/5
5143/5143 [==============================] - 45s 9ms/step - loss: 0.2182 - sparse_categorical_accuracy: 0.9169
Epoch 5/5
5143/5143 [==============================] - 45s 9ms/step - loss: 0.2076 - sparse_categorical_accuracy: 0.9211
Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_3 (Conv2D)           (None, 25, 1, 96)         1632      
                                                                 
 conv2d_4 (Conv2D)           (None, 25, 1, 64)         55360     
                                           

In [40]:
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error
                             ,mean_absolute_error,roc_auc_score,confusion_matrix)
from sklearn import metrics

In [41]:
y_pred = Model.predict(X_validation, verbose = 0)
# predict crisp classes for test set
yhat_classes = np.argmax(y_pred,axis=1)
# reduce to 1d array
yhat_probs = y_pred[:, 0]

 
# accuracy: (tp + tn) / (p + n)
accuracy = accuracy_score(Y_validation, yhat_classes)
print('Accuracy: %f' % accuracy)
# precision tp / (tp + fp)
precision = precision_score(Y_validation, yhat_classes,average='macro')
print('Precision: %f' % precision)
# recall: tp / (tp + fn)
recall = recall_score(Y_validation, yhat_classes,average='macro')
print('Recall: %f' % recall)
# f1: 2 tp / (2 tp + fp + fn)
f1 = f1_score(Y_validation, yhat_classes,average='macro')
print('F1 score: %f' % f1)


# ROC AUC
#auc = roc_auc_score(Y_validation, yhat_probs,multi_class='ovr')
#print('ROC AUC: %f' % auc)
# confusion matrix
matrix = confusion_matrix(Y_validation, yhat_classes)
print(matrix)
# false alaram rate
false_alaram_rate = matrix[1,0]/(matrix[1,0]+matrix[0,0])
print('false alaram rate: %f' % false_alaram_rate)

Accuracy: 0.905514
Precision: 0.853856
Recall: 0.876102
F1 score: 0.858723
[[11862     2     8     2     2]
 [    0  3829  1201    21    31]
 [    0  1587 42888    82  1569]
 [    1    32   328  7434     2]
 [    0  2368   525    13  8490]]
false alaram rate: 0.000000
